# GIAI ĐOẠN 2: TÍNH CHỈ SỐ PHÁP CHỨNG XÁC ĐỊNH 100% & CHUẨN HÓA NGÀNH ICB

**Dự án:** `FinAudit AI — FinForensics` (SE-Hackathon Challenge 2026 — Trường Kinh tế, ĐH Công nghiệp Hà Nội)  
**Mục tiêu Notebook:** Trực quan hóa toàn bộ quy trình làm sạch, tính toán **8 biến Beneish M-Score (1999)**, **4 biến Altman Emerging Market $Z''$-Score (2005, ngưỡng chuẩn `5.85 / 4.35`)**, **7 chỉ số chất lượng lợi nhuận & đặc thù Việt Nam**, kiểm thử Unit Test toán học 100% và chuẩn hóa **Peer-Group Robust Z-Score** theo 16 ngành ICB.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'stage_2_preprocessing_forensics' else Path.cwd()
PROC_DIR = ROOT_DIR / 'data_store' / 'processed'
STAGE_DIR = ROOT_DIR / 'stage_2_preprocessing_forensics'

df = pd.read_parquet(PROC_DIR / 'forensic_features_full.parquet')
bench = pd.read_parquet(PROC_DIR / 'sector_benchmarks_by_year.parquet')
summary = json.loads((PROC_DIR / 'stage2_metrics_summary.json').read_text(encoding='utf-8'))

print(f'Bảng đặc trưng pháp chứng: {df.shape[0]:,} dòng x {df.shape[1]} cột ({df["ticker"].nunique()} DN)')
print(f'Bảng chuẩn ngành ICB     : {bench.shape[0]:,} bản ghi chuẩn ngành')

## Bước 1: Kiểm tra Kết quả 5 Bài Kiểm thử Đơn vị Toán học (Unit Tests)
Xác nhận sai số toán học bằng `0.0000%` trên cả 2 công ty mẫu (Cẩm nang V2) lẫn dữ liệu thực tế `FPT 2025`, không có rò rỉ dữ liệu chéo Lag-1 và không tồn tại giá trị vô cực `+/-inf`.

In [ ]:
display(pd.DataFrame(summary['unit_tests']))

## Bước 2: Phân tích Ma trận Pháp chứng 4 Góc Phần tư (`Beneish × Altman Z''`) & Đối chiếu 9 Doanh nghiệp Tiêu biểu 2025

In [ ]:
df25 = df[df['year'] == 2025].copy()
print('=== Phân bổ 4 Góc Phần tư Năm 2025 ===')
display(df25[df25['beneish_m_score'].notna()]['forensic_quadrant'].value_counts().to_frame('Số doanh nghiệp'))

spot = ['FPT', 'VNM', 'MWG', 'HPG', 'NVL', 'HAG', 'HVN', 'TTF', 'DIG']
cols = ['ticker', 'icb_sector_vi', 'beneish_m_score', 'beneish_flag', 'altman_z_double_prime', 'altman_zone', 'cfo_to_ni_raw', 'tata', 'sensitive_receivables_ratio', 'provision_coverage_ratio']
display(df25[df25['ticker'].isin(spot)].set_index('ticker').reindex(spot)[cols[1:]])

## Bước 3: Kiểm tra Sự Khác biệt Chuẩn Ngành ICB (Tại sao Phải Dùng Peer-Group Robust Z-Score?)
So sánh trung vị ngành năm 2025 và quan sát điểm chuẩn hóa `z_*` của một doanh nghiệp so với chính ngành của nó.

In [ ]:
sec25 = df25.groupby('icb_sector_vi').agg(
    n_co=('ticker', 'nunique'),
    med_beneish=('beneish_m_score', 'median'),
    pct_beneish_flag=('beneish_flag', lambda s: s.mean()*100),
    med_altman_z=('altman_z_double_prime', 'median'),
    med_srr_pct=('sensitive_receivables_ratio', lambda s: s.median()*100),
    med_inv_pct=('inventory_to_assets', lambda s: s.median()*100),
    med_prov_pct=('provision_coverage_ratio', lambda s: s.median()*100),
).sort_values('n_co', ascending=False)
display(sec25)

## Bước 4: Xem Biểu đồ 4 Khung Tổng quan Giai đoạn 2

In [ ]:
from IPython.display import Image, display as ipy_display
ipy_display(Image(filename=str(STAGE_DIR / 'stage2_forensic_distributions.png')))